In [ ]:
import findspark
import pyspark
from jupyter_server.nbconvert.handlers import date_format
from pyspark.sql import SparkSession

findspark.init()

In [30]:
spark = SparkSession.builder.getOrCreate()

In [31]:
registerDF = spark.read.load('sampleData/registerSample.csv',
                             format='csv',
                             header=True,
                             inferSchema=True,
                             sep='\t')

In [32]:
stationsDF = spark.read.load('sampleData/stations.csv',
                             format='csv',
                             header=True,
                             inferSchema=True,
                             sep='\t')

In [33]:
"""
DATAFRAME SOLUTION
"""

'\nDATAFRAME SOLUTION\n'

In [34]:
filteredDF = registerDF.filter("not(used_slots = 0 and free_slots = 0)")

In [35]:
spark.udf.register("get_criticality", lambda free_slots: 1 if free_slots == 0 else 0, pyspark.sql.types.IntegerType())

<function __main__.<lambda>(free_slots)>

In [36]:
selectedDF = filteredDF.selectExpr("station",
                                   "date_format(timestamp, 'EE') as weekday",
                                   "hour(timestamp) as hour",
                                   "get_criticality(free_slots) as critical_slot")

In [37]:
resultDF = selectedDF.groupBy("station", "weekday", "hour").agg({"critical_slot": "avg"})

In [38]:
thresholdedDF = resultDF.filter("avg(critical_slot) > 0.3")

In [39]:
joinedDF = thresholdedDF.join(stationsDF, thresholdedDF.station == stationsDF.id).withColumnRenamed("avg(critical_slot)", "criticality")

In [40]:
selectedDF = joinedDF.select("station",
                             "weekday",
                             "hour",
                             "criticality",
                             "latitude",
                             "longitude")

In [41]:
sortedDF = selectedDF.sort(selectedDF.criticality.desc(), selectedDF.station.asc(), selectedDF.weekday.asc(), selectedDF.hour.asc())
sortedDF.show()

+-------+-------+----+-------------------+---------+---------+
|station|weekday|hour|        criticality| latitude|longitude|
+-------+-------+----+-------------------+---------+---------+
|      1|    Thu|   0| 0.4581005586592179|41.397978| 2.180019|
|      1|    Thu|   1| 0.4329608938547486|41.397978| 2.180019|
|      1|    Sun|   4|  0.403899721448468|41.397978| 2.180019|
|      1|    Wed|  23|  0.388086642599278|41.397978| 2.180019|
|      1|    Thu|   2|0.38341968911917096|41.397978| 2.180019|
|      1|    Tue|   0| 0.3743016759776536|41.397978| 2.180019|
|      1|    Wed|  22|0.37122557726465366|41.397978| 2.180019|
|      1|    Mon|   1| 0.3659217877094972|41.397978| 2.180019|
|      1|    Wed|   0|0.34108527131782945|41.397978| 2.180019|
|      1|    Mon|   0| 0.3380281690140845|41.397978| 2.180019|
|      1|    Tue|   1| 0.3352272727272727|41.397978| 2.180019|
|      1|    Fri|   0| 0.3307291666666667|41.397978| 2.180019|
|      1|    Sun|   3|0.32590529247910865|41.397978| 2.

In [42]:
#sortedDF.write.csv('result', header=True, sep='\t')

In [43]:
"""
SQL QUERY SOLUTION
"""

'\nSQL QUERY SOLUTION\n'

In [44]:
registerDF.createOrReplaceTempView("register")
stationsDF.createOrReplaceTempView("stations")

In [45]:
selectedDF = spark.sql("SELECT station, "
                       "date_format(timestamp, 'EE') as weekday, "
                       "hour(timestamp) as hour, "
                       "AVG(get_criticality(free_slots)) as criticality "
                       "FROM register "
                       "WHERE NOT(used_slots = 0 AND free_slots = 0) "
                       "GROUP BY station, weekday, hour "
                       "HAVING criticality > 0.3 ")
selectedDF.show()
selectedDF.createOrReplaceTempView("selected")

+-------+-------+----+-------------------+
|station|weekday|hour|        criticality|
+-------+-------+----+-------------------+
|      1|    Fri|   1| 0.3191489361702128|
|      1|    Tue|   1| 0.3352272727272727|
|      1|    Thu|   3|0.31733333333333336|
|      1|    Thu|   2|0.38341968911917096|
|      1|    Wed|   0|0.34108527131782945|
|      1|    Tue|   0| 0.3743016759776536|
|      1|    Mon|   1| 0.3659217877094972|
|      1|    Sun|   4|  0.403899721448468|
|      1|    Mon|   0| 0.3380281690140845|
|      1|    Sun|   7|0.30560578661844484|
|      1|    Thu|   0| 0.4581005586592179|
|      1|    Wed|  23|  0.388086642599278|
|      1|    Fri|   0| 0.3307291666666667|
|      2|    Mon|   1|0.31564245810055863|
|      1|    Wed|  22|0.37122557726465366|
|      1|    Thu|   1| 0.4329608938547486|
|      1|    Sun|   3|0.32590529247910865|
+-------+-------+----+-------------------+



In [46]:
joinedDF = spark.sql("SELECT station, weekday, hour, criticality, latitude, longitude "
                     "FROM selected, stations "
                     "WHERE station = id "
                     "ORDER BY criticality DESC, station ASC, weekday ASC, hour ASC")
joinedDF.show()

+-------+-------+----+-------------------+---------+---------+
|station|weekday|hour|        criticality| latitude|longitude|
+-------+-------+----+-------------------+---------+---------+
|      1|    Thu|   0| 0.4581005586592179|41.397978| 2.180019|
|      1|    Thu|   1| 0.4329608938547486|41.397978| 2.180019|
|      1|    Sun|   4|  0.403899721448468|41.397978| 2.180019|
|      1|    Wed|  23|  0.388086642599278|41.397978| 2.180019|
|      1|    Thu|   2|0.38341968911917096|41.397978| 2.180019|
|      1|    Tue|   0| 0.3743016759776536|41.397978| 2.180019|
|      1|    Wed|  22|0.37122557726465366|41.397978| 2.180019|
|      1|    Mon|   1| 0.3659217877094972|41.397978| 2.180019|
|      1|    Wed|   0|0.34108527131782945|41.397978| 2.180019|
|      1|    Mon|   0| 0.3380281690140845|41.397978| 2.180019|
|      1|    Tue|   1| 0.3352272727272727|41.397978| 2.180019|
|      1|    Fri|   0| 0.3307291666666667|41.397978| 2.180019|
|      1|    Sun|   3|0.32590529247910865|41.397978| 2.